#Cleaning: "In which country are you located?"
working notebook for standardizing the country column.

In [1]:
import pandas as pd
import numpy as np

In [2]:

df = pd.read_excel('Responses(1).xlsx')
df.shape

(685, 57)

In [3]:
#(Idrees)- the column we're cleaning
col = 'In which country are you located?'

In [4]:
#(Idrees)check the raw values
df[col].value_counts(dropna=False)

In which country are you located?
Netherlands               123
Germany                   118
Spain                      85
Greece                     58
Netherlands                50
                         ... 
India                       1
Amsterxam                   1
Uithoorn                    1
Amsterdam                   1
Netherlands, Amsterdam      1
Name: count, Length: 72, dtype: int64

In [5]:
#(Idrees)number of unique values and missing count
print('Unique values:', df[col].nunique(dropna=True))
print('Missing:', df[col].isna().sum())

Unique values: 71
Missing: 3


In [6]:
#(Idrees)basic first cleanup: fill NaN with 'Unknown', convert to string, and strip whitespace 
df[col].astype(str).str.strip()

0          Netherlands
1          Netherlands
2          Netherlands
3                Malta
4      The Netherlands
            ...       
680        Netherlands
681        Netherlands
682            Belgium
683        Netherlands
684        Netherlands
Name: In which country are you located?, Length: 685, dtype: str

In [7]:
#(Idrees) fix all Netherlands variants to "The Netherlands"
netherlands_variants = [
    'netherlands', 'the netherlands', 'nederland', 'niederlande', 'holland', 'nl',
    'netherland', 'the netherlans', 'netherworld',
    'amsterdam', 'amsterxam', 'uithoorn', 'netherlands, amsterdam',
]

#compare on a stripped + lowercased version so spacing/case don't matter
normalized = df[col].astype(str).str.strip().str.lower()
df.loc[normalized.isin(netherlands_variants), col] = 'The Netherlands'

#check the result
df[col].value_counts(dropna=False)
print(normalized.isin(netherlands_variants).sum(), 'rows updated to "The Netherlands"')

298 rows updated to "The Netherlands"


In [8]:
#(Idrees) clean all the other countries the same way, using a normalized then clean-name map.
#keys are stripped + lowercased, so case/spacing variants are handled automatically.
country_map = {
    'germany': 'Germany', 'deutschland': 'Germany', 'german': 'Germany',
    'spain': 'Spain', 'españa': 'Spain', 'espana': 'Spain',
    'greece': 'Greece', 'greece.': 'Greece',
    'turkiye': 'Turkey', 'türkiye': 'Turkey', 'تركيا': 'Turkey',
    'egypt': 'Egypt', 'cairo': 'Egypt',
    'united kingdom': 'United Kingdom', 'uk': 'United Kingdom', 'england': 'United Kingdom',
    'usa': 'United States', 'united states': 'United States',
    'malta': 'Malta', 'france': 'France', 'sweden': 'Sweden', 'austria': 'Austria',
    'morocco': 'Morocco', 'belgium': 'Belgium', 'canada': 'Canada', 'portugal': 'Portugal',
    'brazil': 'Brazil', 'jordan': 'Jordan', 'norway': 'Norway', 'india': 'India',
    'new zealand': 'New Zealand', 'poland': 'Poland', 'south africa': 'South Africa',
    'switzerland': 'Switzerland', 'italy': 'Italy', 'hong kong': 'Hong Kong',
    'singapore': 'Singapore',
}

normalized = df[col].astype(str).str.strip().str.lower()
mapped = normalized.map(country_map)
#where we have a mapping use it; otherwise keep the current value (Netherlands, blanks, or whatever else is there)
df[col] = mapped.fillna(df[col])

#check the result
df[col].value_counts(dropna=False)

In which country are you located?
The Netherlands    298
Germany            151
Spain              113
Greece              76
Austria              4
United Kingdom       4
Malta                3
NaN                  3
Canada               3
France               3
Sweden               3
United States        2
Egypt                2
Belgium              2
Turkey               2
Morocco              2
Portugal             2
New Zealand          1
Singapore            1
South Africa         1
Switzerland          1
Poland               1
Italy                1
Hong Kong            1
Spiliotopoulos       1
Jordan               1
Brazil               1
Norway               1
India                1
Name: count, dtype: int64

In [9]:
#save to a new file
df.to_excel('Responses(1)_cleaned.xlsx', index=False)
print('Saved -> Responses(1)_cleaned.xlsx')

Saved -> Responses(1)_cleaned.xlsx
